# **Penting**

*   Pada Notebook ini, Anda hanya perlu mengerjakan code pada bagian **logic.py** saja. Anda tidak diwajibkan untuk mengubah atau menambahkan **app.py** yang digunakan untuk membangun interface Streamlit.
*   Namun, jika Anda memiliki preferensi lain atau ingin mengubah struktur code pada logic ataupun pada interface Streamlit, itu **DIPERSILAHKAN** saja, tetapi pastikan untuk memenuhi kriteria yang telah ditetapkan pada intruksi submission
*   Jika Anda tidak ingin mengubah apapun dan ingin mengikuti template, tugas Anda hanyalah melengkapi code yang rumpang pada bagian yang sudah ditandai "________" saja.



**Nama: Kendrick Filbert** — Lunar Studio. Adaptasi template dengan lazy model loading, input validation, hasil persisten, dan ekspor PNG. Gunakan T4 GPU untuk inference.

# **Prepare Dependencies**

In [ ]:
from pathlib import Path
Path('requirements.txt').write_text("# Python 3.11/3.12. Keep Colab's preinstalled CUDA-enabled torch.\ntorch>=2.4,<3\ndiffusers==0.35.1\ntransformers==4.56.2\naccelerate==1.10.1\nhuggingface-hub==0.34.4\nsafetensors==0.6.2\nnumpy==1.26.4\nPillow==10.4.0\nmatplotlib==3.9.2\npandas==2.2.3\nstreamlit==1.32.2\nstreamlit-drawable-canvas==0.9.3\npyngrok==7.2.12\nsentencepiece==0.2.1\nprotobuf>=3.20,<5\nnbformat==5.10.4\nnbclient==0.10.2\nipykernel>=6.29,<7\nimageio-ffmpeg==0.6.0\n", encoding='utf-8')
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print('Dependensi terpasang. Jika diminta restart oleh Colab, restart lalu lanjut dari sel berikutnya.')

In [ ]:
import os, sys, subprocess, time, urllib.request
from pathlib import Path
from pyngrok import ngrok
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan GPU T4 di Runtime > Change runtime type.')
print('GPU:', torch.cuda.get_device_name(0))

# **Streamlit**

## logic.py (**Basic**)

In [ ]:
%%writefile logic.py
"""Generative Image Suite — Kendrick Filbert.

SD 1.5 submission backend. Model imports are lazy so geometry tests need no GPU.
White mask pixels are editable; black pixels are preserved exactly.
"""
import gc
import threading
from contextlib import contextmanager
from functools import lru_cache
from PIL import Image, ImageDraw, ImageFilter, ImageOps
import numpy as np

BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
INPAINT_MODEL = "stable-diffusion-v1-5/stable-diffusion-inpainting"
PROMPT = ("2D digital illustration of a cute astronaut in a white spacesuit, "
          "standing on the moon, full body on the left, Earth in the starry sky, "
          "empty lunar ground on the right, clean outlines, colorful digital art")
NEGATIVE_PROMPT = ("photorealistic, realistic, photograph, 3d render, messy, blurry, "
                   "low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration")
SCHEDULERS = ("Euler A", "DPM++", "DDIM")
INFERENCE_LOCK = threading.RLock()


def flush_memory():
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _configure(pipe):
    import torch
    pipe.enable_vae_slicing()
    pipe.enable_attention_slicing()
    # Keep weights on CPU between calls; activate only the current pipeline.
    if not torch.cuda.is_available():
        pipe.to("cpu")
    return pipe


@lru_cache(maxsize=2)
def get_pipeline(task="generation"):
    import torch
    from diffusers import StableDiffusionPipeline, StableDiffusionInpaintPipeline
    if task not in ("generation", "inpainting"):
        raise ValueError("Task harus generation atau inpainting.")
    cls = StableDiffusionPipeline if task == "generation" else StableDiffusionInpaintPipeline
    model = BASE_MODEL if task == "generation" else INPAINT_MODEL
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    pipe = cls.from_pretrained(model, torch_dtype=dtype, use_safetensors=True)
    return _configure(pipe)


def load_models_cached():
    """Compatibility with the template. Prefer lazy get_pipeline in the UI."""
    return get_pipeline("generation"), get_pipeline("inpainting")


@contextmanager
def active_pipeline(pipe):
    import torch
    with INFERENCE_LOCK:
        try:
            if torch.cuda.is_available():
                pipe.to("cuda")
            yield pipe
        finally:
            if torch.cuda.is_available():
                pipe.to("cpu")
            flush_memory()


def load_scheduler(pipe, scheduler_name):
    from diffusers import (EulerAncestralDiscreteScheduler,
                           DPMSolverMultistepScheduler, DDIMScheduler)
    classes = {"Euler A": EulerAncestralDiscreteScheduler,
               "DPM++": DPMSolverMultistepScheduler, "DDIM": DDIMScheduler}
    if scheduler_name not in classes:
        raise ValueError(f"Scheduler tidak dikenal: {scheduler_name}")
    if not hasattr(pipe, "_original_scheduler_config"):
        pipe._original_scheduler_config = dict(pipe.scheduler.config)
    options = {"algorithm_type": "dpmsolver++", "use_karras_sigmas": True} if scheduler_name == "DPM++" else {}
    pipe.scheduler = classes[scheduler_name].from_config(pipe._original_scheduler_config, **options)
    return pipe


set_scheduler = load_scheduler


def _validate(prompt, seed, steps, cfg, num_images=1):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Prompt tidak boleh kosong.")
    if not 0 <= int(seed) <= 2**32 - num_images:
        raise ValueError("Seed di luar rentang uint32.")
    if not 1 <= int(steps) <= 100 or not 1 <= int(num_images) <= 4:
        raise ValueError("Steps 1–100 dan jumlah gambar 1–4.")
    if not 1 <= float(cfg) <= 20:
        raise ValueError("CFG harus 1–20.")


def _check_tokens(pipe, prompt):
    tokens = pipe.tokenizer(prompt, truncation=False).input_ids
    if len(tokens) > pipe.tokenizer.model_max_length:
        raise ValueError("Prompt terlalu panjang untuk CLIP. Ringkas menjadi maksimal 75 token isi.")


def generate_image(pipe, prompt, neg_prompt, seed, steps, cfg,
                   num_images=1, scheduler_name="Euler A"):
    import torch
    _validate(prompt, seed, steps, cfg, num_images)
    _check_tokens(pipe, prompt)
    _check_tokens(pipe, neg_prompt)
    # One call, true batch inference; a fresh generator per output.
    generators = [torch.Generator(device="cpu").manual_seed(int(seed) + i)
                  for i in range(int(num_images))]
    with active_pipeline(pipe), torch.inference_mode():
        load_scheduler(pipe, scheduler_name)
        output = pipe(prompt=prompt, negative_prompt=neg_prompt,
                      generator=generators, num_images_per_prompt=int(num_images),
                      num_inference_steps=int(steps), guidance_scale=float(cfg),
                      width=512, height=512)
    if any(getattr(output, "nsfw_content_detected", None) or []):
        raise ValueError("Safety checker memblokir salah satu hasil; ubah prompt.")
    return output.images


def generate_simple_image(prompt, negative_prompt=NEGATIVE_PROMPT, seed=222):
    return generate_image(get_pipeline(), prompt, negative_prompt, seed, 30, 7.5)[0]


def generate_advanced_image(prompt, negative_prompt=NEGATIVE_PROMPT, seed=222,
                            guidance_scale=9.0, num_inference_step=40):
    return generate_image(get_pipeline(), prompt, negative_prompt, seed,
                          num_inference_step, guidance_scale)[0]


def manual_mask(image, box=(285, 230, 485, 450)):
    """Hardcoded starting rectangle for 512px output; adjust after visual review."""
    mask = Image.new("L", image.size, 0)
    ImageDraw.Draw(mask).rectangle(box, fill=255)
    return mask


def normalize_image(image):
    image = ImageOps.exif_transpose(image).convert("RGB")
    w, h = image.size
    if w < 64 or h < 64 or max(w, h) > 1536:
        raise ValueError("Ukuran gambar harus 64–1536 piksel per sisi.")
    if w % 8 or h % 8:
        raise ValueError("Ukuran gambar harus kelipatan 8; jangan merusak geometri mask.")
    return image


def run_inpainting(pipe, image, mask, prompt, strength=1.0, seed=9,
                   steps=40, cfg=9.0, negative_prompt=NEGATIVE_PROMPT,
                   feather=4):
    import torch
    _validate(prompt, seed, steps, cfg)
    if not 0 < strength <= 1 or int(steps * strength) < 1:
        raise ValueError("Strength harus menghasilkan minimal satu denoising step.")
    _check_tokens(pipe, prompt)
    image = normalize_image(image)
    mask = mask.convert("L").resize(image.size, Image.Resampling.NEAREST)
    mask = mask.point(lambda p: 255 if p >= 128 else 0)
    if mask.getbbox() is None:
        raise ValueError("Mask kosong: tandai area yang akan diedit.")
    with active_pipeline(pipe), torch.inference_mode():
        load_scheduler(pipe, "DPM++")
        output = pipe(prompt=prompt, negative_prompt=negative_prompt,
                      image=image, mask_image=mask, width=image.width, height=image.height,
                      strength=float(strength), guidance_scale=float(cfg),
                      num_inference_steps=int(steps),
                      generator=torch.Generator("cpu").manual_seed(int(seed)))
    if any(getattr(output, "nsfw_content_detected", None) or []):
        raise ValueError("Safety checker memblokir hasil edit.")
    # Blur only inside editable pixels, preserving every black-mask pixel.
    alpha = np.asarray(mask.filter(ImageFilter.GaussianBlur(feather))).copy()
    alpha[np.asarray(mask) == 0] = 0
    return Image.composite(output.images[0], image, Image.fromarray(alpha))


def inpaint_engine(image, mask, prompt, **kwargs):
    return run_inpainting(get_pipeline("inpainting"), image, mask, prompt, **kwargs)


def prepare_outpainting(image, expand_pixels=128, direction="all", overlap=24):
    """Expand geometrically; overlap lets diffusion blend the original boundary."""
    image = normalize_image(image)
    if direction not in ("left", "right", "top", "bottom", "all"):
        raise ValueError("Arah perluasan tidak valid.")
    if expand_pixels < 8 or expand_pixels % 8:
        raise ValueError("Ekspansi harus kelipatan 8 dan minimal 8 piksel.")
    if not 0 <= overlap < min(image.size) // 2:
        raise ValueError("Overlap terlalu besar.")
    left = expand_pixels if direction in ("left", "all") else 0
    right = expand_pixels if direction in ("right", "all") else 0
    top = expand_pixels if direction in ("top", "all") else 0
    bottom = expand_pixels if direction in ("bottom", "all") else 0
    w, h = image.size
    if max(w + left + right, h + top + bottom) > 1536:
        raise ValueError("Kanvas melebihi batas 1536px; gunakan gambar lebih kecil.")
    # Edge extension avoids a solid-color border interpreted as a picture frame.
    pixels = np.pad(np.asarray(image), ((top, bottom), (left, right), (0, 0)), mode="edge")
    canvas = Image.fromarray(pixels).filter(ImageFilter.GaussianBlur(18))
    canvas.paste(image, (left, top))
    mask = Image.new("L", canvas.size, 255)
    keep = (left + (overlap if left else 0), top + (overlap if top else 0),
            left + w - (overlap if right else 0), top + h - (overlap if bottom else 0))
    # PIL rectangle includes the final coordinate; subtract one for exclusive bounds.
    ImageDraw.Draw(mask).rectangle((keep[0], keep[1], keep[2]-1, keep[3]-1), fill=0)
    return canvas, mask


@lru_cache(maxsize=1)
def load_segmentation():
    from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
    model_id = "CIDAS/clipseg-rd64-refined"
    return (CLIPSegProcessor.from_pretrained(model_id),
            CLIPSegForImageSegmentation.from_pretrained(model_id).eval())


def segmentation_mask(image, text="astronaut", threshold=0.45):
    import torch
    if not 0 < threshold < 1:
        raise ValueError("Threshold harus antara 0 dan 1.")
    processor, model = load_segmentation()
    inputs = processor(text=[text], images=[image.convert("RGB")],
                       padding=True, return_tensors="pt")
    with torch.inference_mode():
        logits = model(**inputs).logits.reshape(1, 1, 352, 352)
        probabilities = torch.nn.functional.interpolate(
            logits, size=(image.height, image.width), mode="bilinear", align_corners=False
        ).sigmoid()[0, 0].numpy()
    mask = Image.fromarray((probabilities >= threshold).astype("uint8") * 255)
    fraction = np.count_nonzero(np.asarray(mask)) / (image.width * image.height)
    if not 0.005 <= fraction <= 0.8:
        raise ValueError(f"Mask segmentasi tidak layak ({fraction:.1%}); sesuaikan teks/threshold.")
    return mask.filter(ImageFilter.MaxFilter(7)), probabilities


def refine_image(image, prompt, negative_prompt=NEGATIVE_PROMPT, seed=222,
                 strength=0.25, steps=40, cfg=7.5):
    import torch
    from diffusers import StableDiffusionImg2ImgPipeline
    base = get_pipeline()
    # Share the base components; never load a second copy of the checkpoint.
    with INFERENCE_LOCK:
        refiner = StableDiffusionImg2ImgPipeline(**base.components)
        _validate(prompt, seed, steps, cfg)
        if not 0 < strength <= 0.4 or int(steps * strength) < 1:
            raise ValueError("Refiner strength harus kecil: 0 < strength <= 0.4.")
        with active_pipeline(refiner), torch.inference_mode():
            load_scheduler(refiner, "DPM++")
            output = refiner(prompt=prompt, negative_prompt=negative_prompt,
                             image=image.convert("RGB"), strength=strength,
                             num_inference_steps=steps, guidance_scale=cfg,
                             generator=torch.Generator("cpu").manual_seed(seed))
        if any(getattr(output, "nsfw_content_detected", None) or []):
            raise ValueError("Safety checker memblokir hasil refiner.")
        return output.images[0]

In [ ]:
import importlib, logic
importlib.reload(logic)
assert callable(logic.generate_image)
print('Basic: generate_image, negative prompt, seed, steps, CFG tersedia.')

## logic.py (**Skilled**)

In [ ]:
assert logic.SCHEDULERS == ('Euler A', 'DPM++', 'DDIM')
logic.flush_memory()
print('Skilled: true batch 1–4, tiga scheduler, dan Clear Memory tersedia.')

## logic.py (**Advanced**)

In [ ]:
from PIL import Image
canvas, mask = logic.prepare_outpainting(Image.new('RGB', (512,512)), 128, 'all')
assert canvas.size == (768,768) and mask.getpixel((384,384)) == 0
print('Advanced: canvas mask, inpainting, outpainting, dan zoom out tersedia.')

## app.py
Anda **TIDAK perlu mengubah atau menambahkan** apapun pada file **app.py** ini, cukup **jalankan** saja.

Versi berikut menyesuaikan template untuk menjaga hasil setelah rerun, lazy model loading, validasi mask, dan download PNG. Struktur tab Generation/Edit tetap dipertahankan.

In [ ]:
%%writefile app.py
"""Streamlit interface adapted from the supplied BFGAI template."""
import io
import json
import time
import streamlit as st
import numpy as np
from PIL import Image, ImageFilter
from streamlit_drawable_canvas import st_canvas
import logic

st.set_page_config(page_title="Lunar Studio · Kendrick Filbert", page_icon="🌕", layout="wide")
st.title("Lunar Studio")
st.caption("Generative Image Suite · Kendrick Filbert · Stable Diffusion 1.5")

@st.cache_resource
def get_model(task):
    return logic.get_pipeline(task)

def png_bytes(image):
    output = io.BytesIO()
    image.save(output, format="PNG")
    return output.getvalue()

def show_error(exc):
    if "out of memory" in str(exc).lower():
        st.error("Memori GPU tidak cukup. Kurangi batch atau ukuran ekspansi, lalu coba kembali.")
    else:
        st.error(f"Proses belum berhasil: {exc}")

with st.sidebar:
    st.header("Generation settings")
    steps = st.slider("Inference steps", 5, 50, 30)
    cfg = st.slider("Guidance scale (CFG)", 1.0, 20.0, 7.5, 0.5)
    seed = int(st.number_input("Seed", min_value=0, max_value=4294967292, value=222))
    scheduler_name = st.selectbox("Scheduler", logic.SCHEDULERS)
    num_images = st.slider("Jumlah gambar", 1, 4, 1)
    st.caption("Seed tiap gambar = seed awal + indeks. Empat gambar diproses dalam satu batch.")
    if st.button("Clear Memory"):
        logic.flush_memory()
        st.success("Cache alokasi yang tidak dipakai dibersihkan.")

tab_gen, tab_edit = st.tabs(["Generation", "Edit"])
with tab_gen:
    inputs, outputs = st.columns(2, gap="large")
    with inputs:
        st.subheader("Describe your scene")
        with st.form("generation"):
            prompt = st.text_area("Prompt", logic.PROMPT, height=150)
            negative = st.text_area("Negative prompt", logic.NEGATIVE_PROMPT, height=100)
            generate = st.form_submit_button("Generate", type="primary")
        if generate:
            try:
                start = time.perf_counter()
                with st.spinner("Menghasilkan gambar dengan Stable Diffusion…"):
                    images = logic.generate_image(get_model("generation"), prompt,
                        negative, seed, steps, cfg, num_images, scheduler_name)
                st.session_state.generated_images = images
                st.session_state.current_image = images[0]
                st.session_state.canvas_revision = st.session_state.get("canvas_revision", 0) + 1
                st.session_state.generation_meta = dict(model=logic.BASE_MODEL,
                    prompt=prompt, negative_prompt=negative, seed=seed, steps=steps,
                    cfg=cfg, scheduler=scheduler_name, num_images=num_images,
                    seconds=round(time.perf_counter()-start, 2))
            except Exception as exc:
                show_error(exc)
    with outputs:
        st.subheader("Visual output")
        images = st.session_state.get("generated_images", [])
        if images:
            cols = st.columns(2) if len(images) > 1 else [st.container()]
            meta = st.session_state.generation_meta
            for i, image in enumerate(images):
                with cols[i % len(cols)]:
                    st.image(image, caption=f"Gambar {i+1} · seed {meta['seed']+i}", use_column_width=True)
                    if st.button(f"Edit gambar {i+1}", key=f"select_{i}"):
                        st.session_state.current_image = image
                        st.session_state.canvas_revision = st.session_state.get("canvas_revision", 0) + 1
                        st.success("Gambar dipilih. Buka tab Edit.")
                    st.download_button("Download PNG", png_bytes(image),
                        file_name=f"generation_{i+1}.png", mime="image/png", key=f"download_{i}")
            st.caption(f"Inference selesai dalam {meta['seconds']} detik.")
            st.download_button("Download parameter", json.dumps(meta, indent=2),
                               "generation_metadata.json", "application/json")
        else:
            st.info("Isi prompt lalu klik Generate. Hasil ditampilkan di sini.")

with tab_edit:
    source = st.session_state.get("current_image")
    if source is None:
        st.info("Buat gambar pada tab Generation terlebih dahulu.")
    else:
        mode = st.radio("Mode edit", ["Inpainting", "Outpainting / Zoom Out"], horizontal=True)
        left, right = st.columns(2, gap="large")
        with left:
            st.subheader("Gambar sumber")
            if mode == "Inpainting":
                scale = min(1.0, 512 / max(source.size))
                display_size = (round(source.width*scale), round(source.height*scale))
                preview = source.resize(display_size, Image.Resampling.LANCZOS)
                st.caption("Sapukan kuas putih di area yang ingin diubah.")
                drawing = st_canvas(fill_color="rgba(255,255,255,1)", stroke_width=24,
                    stroke_color="#FFFFFF", background_image=preview,
                    update_streamlit=True, height=preview.height, width=preview.width,
                    drawing_mode="freedraw", key=f"mask_{st.session_state.get('canvas_revision', 0)}")
            else:
                st.image(source, use_column_width=True)
        with right:
            st.subheader("Edit settings")
            with st.form("edit"):
                edit_prompt = st.text_area("Prompt edit", "2D illustration of a broken satellite with blue solar panels on the moon, Earth in the starry sky")
                edit_seed = int(st.number_input("Seed edit", min_value=0, max_value=4294967295, value=9))
                strength = st.slider("Strength", 0.1, 1.0, 1.0, 0.05)
                if mode != "Inpainting":
                    direction = st.selectbox("Arah perluasan", ["all", "right", "left", "top", "bottom"])
                    expansion = st.slider("Ekspansi (piksel)", 64, 256, 128, 64)
                apply = st.form_submit_button("Apply edit", type="primary")
            if apply:
                try:
                    if mode == "Inpainting":
                        if drawing.image_data is None:
                            raise ValueError("Gambar mask terlebih dahulu.")
                        alpha = drawing.image_data[:, :, 3]
                        if not np.any(alpha > 0):
                            raise ValueError("Mask masih kosong.")
                        mask = Image.fromarray((alpha > 0).astype("uint8") * 255)
                        mask = mask.resize(source.size, Image.Resampling.NEAREST).filter(ImageFilter.MaxFilter(9))
                        canvas = source
                    else:
                        canvas, mask = logic.prepare_outpainting(source, expansion, direction)
                    with st.spinner("Menjalankan inpainting…"):
                        result = logic.run_inpainting(get_model("inpainting"), canvas, mask,
                            edit_prompt, strength, seed=edit_seed, steps=steps, cfg=cfg)
                    st.session_state.before_edit = source
                    st.session_state.current_image = result
                    st.session_state.last_mask = mask
                    st.session_state.canvas_revision = st.session_state.get("canvas_revision", 0) + 1
                    st.rerun()
                except Exception as exc:
                    show_error(exc)
        if "before_edit" in st.session_state:
            before, after = st.columns(2)
            before.image(st.session_state.before_edit, caption="Before", use_column_width=True)
            after.image(st.session_state.current_image, caption="After", use_column_width=True)
            st.download_button("Download hasil edit", png_bytes(st.session_state.current_image),
                               "edited_image.png", "image/png")
            with st.expander("Lihat mask terakhir"):
                st.image(st.session_state.last_mask, width=256)

# **Menggunakan *Ngrok* Untuk Deployment**

## **Konfigurasi Autentikasi Ngrok dan Menjalankan Aplikasi Streamlit**
Cell ini digunakan untuk mengatur *authentication token Ngrok* dan menjalankan aplikasi Streamlit secara lokal. Token diperlukan agar *Ngrok* dapat membuat tunnel dengan akun pengguna. Setelah token diatur, aplikasi Streamlit dijalankan menggunakan subprocess sehingga server lokal aktif di background.

Apabila Anda belum mengetahui cara mendapatkan *authentication token Ngrok* milik Anda sendiri, silahkan baca pada bagian **tips and tricks submission**.

In [ ]:
# Ngrok opsional. Simpan NGROK_AUTHTOKEN di Colab Secrets, jangan hardcode.
auth_token = os.environ.get('NGROK_AUTHTOKEN')
try:
    from google.colab import userdata
    if not auth_token:
        try: auth_token = userdata.get('NGROK_AUTHTOKEN')
        except Exception: pass
except ImportError:
    pass
if auth_token:
    ngrok.set_auth_token(auth_token)
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate(); streamlit_process.wait(timeout=15)
log_file = open('streamlit.log', 'w', encoding='utf-8')
streamlit_process = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py',
    '--server.port=8501', '--server.address=0.0.0.0', '--server.headless=true',
    '--browser.gatherUsageStats=false'], stdout=log_file, stderr=subprocess.STDOUT)
for attempt in range(60):
    if streamlit_process.poll() is not None:
        raise RuntimeError(Path('streamlit.log').read_text())
    try:
        with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=2) as r:
            if r.status == 200: break
    except Exception: time.sleep(1)
else: raise RuntimeError('Streamlit belum sehat; lihat streamlit.log')
print('Streamlit health check: OK')

## **Membuat Public URL**
Cell ini membuat *tunnel Ngrok* ke port lokal tempat Streamlit berjalan (default: 8501). *Ngrok* kemudian menghasilkan public URL yang bisa diakses dari internet, sehingga aplikasi Streamlit dapat dibuka tanpa harus berada di jaringan lokal yang sama.

In [ ]:
from IPython.display import display, HTML
if auth_token:
    tunnel = ngrok.connect(8501)
    public_url = tunnel.public_url
    display(HTML(f'<a href="{public_url}" target="_blank">Buka Lunar Studio</a>'))
else:
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(8501)
        print('Menggunakan proxy Colab tanpa token Ngrok; akses terikat sesi Colab.')
    except ImportError:
        print('Buka http://localhost:8501')
print('Rekam 60–120 detik: prompt, parameter, klik Generate, hasil nyata, lalu Edit jika tersedia.')
print('Simpan rekaman sebagai video_demo_aplikasi_BFGAI.mp4. Download notebook dengan output.')

Apabila Anda mengalami limit endpoint usage pada Ngrok, jalankan hidden cell di bawah ini!

## **Menutup Semua Tunnel Ngrok yang Aktif**
Cell ini digunakan untuk menghentikan seluruh koneksi Ngrok yang masih aktif.

Jalankan penutupan hanya sesudah demo selesai. Default False menjaga aplikasi aktif saat Run all.

In [ ]:
STOP_SERVICES = False
if STOP_SERVICES:
    if 'tunnel' in globals(): ngrok.disconnect(tunnel.public_url)
    if 'streamlit_process' in globals() and streamlit_process.poll() is None:
        streamlit_process.terminate(); streamlit_process.wait(timeout=15)
    print('Layanan milik notebook ini dihentikan.')
else:
    print('Aplikasi tetap berjalan. Ubah STOP_SERVICES=True setelah demo selesai.')